In [32]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("vehicleprice.csv")

df = df.drop('Location', axis=1)
df = df.drop('Title', axis=1)
df.dropna(subset=['Brand', 'Price','Doors','Seats', 'Car/Suv','BodyType', 'Kilometres'], inplace=True)

rows = df[df['UsedOrNew'] == 'DEMO'].index
df.drop(index=rows, inplace=True)

df['Kilometres'] = pd.to_numeric(df['Kilometres'], errors='coerce')
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')



encoder = OneHotEncoder(sparse_output=False)

one_hot_encoded = encoder.fit_transform(df[['Transmission']])

encoded_Transmission = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(['Transmission']))

df = pd.concat([df,encoded_Transmission],axis=1)

df = df.drop('Transmission', axis=1)

df['FuelConsumption_new'] = df['FuelConsumption'].str.extract(r'(\d+\.?\d*)').astype(float)
df['Engine_new'] = df['Engine'].str.extract(r'(\d+\.?\d*)').astype(float)
df['Cylinders_new'] = df['CylindersinEngine'].str.extract(r'(\d+)').astype(float)
df['Doors_new'] = df['Doors'].str.extract(r'(\d+)').astype(float)
df['Seats_new'] = df['Seats'].str.extract(r'(\d+)').astype(float)


categorical_cols = ['Brand', 'Model', 'Car/Suv', 'UsedOrNew',
                    'DriveType', 'FuelType', 'ColourExtInt', 'BodyType']

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    
df = df.drop(['FuelConsumption', 'Engine', 'CylindersinEngine', 'Doors', 'Seats', 'Brand', 'Model', 'Car/Suv', 'UsedOrNew', 'DriveType', 'FuelType', 
              'ColourExtInt', 'BodyType'], axis=1)
        
df.isna().sum()
df.dropna(subset=['Year', 'Price','Kilometres','Transmission_-'], inplace=True)

y = df['Price']
X = df.drop('Price', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = lgb.LGBMRegressor()
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print("MAE:", mae)



[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000545 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1025
[LightGBM] [Info] Number of data points in the train set: 10254, number of used features: 17
[LightGBM] [Info] Start training from score 33537.373805
MAE: 5559.96372258756
    Actual Price  Predicted Price         Error
0        48990.0     35931.053192  13058.946808
1        56585.0     57383.211971    798.211971
2        39999.0     35102.841073   4896.158927
3        36990.0     36272.507669    717.492331
4        18990.0     16859.146479   2130.853521
5        41485.0     41424.922167     60.077833
6        19990.0     18211.810814   1778.189186
7        41999.0     72578.778380  30579.778380
8        69990.0     69981.206341      8.793659
9        53990.0     51441.236750   2548.763250
10       50705.0     44280.403693   6424